# Week 02 — Phone Repair FAQ Semantic Search

**Goal:** Find FAQ entries that are similar in meaning to a question.

This notebook demonstrates loading CSV data, cleaning it, making text chunks, creating embeddings, storing them in Chroma, and retrieving up to three matches. The FAQ answers are sample learning material, not professional repair diagnoses.

## 1. Install and import the libraries

In [ ]:
%pip -q install pandas chromadb sentence-transformers

In [ ]:
from pathlib import Path
import pandas as pd
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

## 2. Upload and load the CSV

Run the upload cell and select `faqs.csv` from your extracted `week-02/data` folder.

In [ ]:
from google.colab import files
uploaded_files = files.upload()

In [ ]:
CSV_FILE = Path("faqs.csv")
faqs = pd.read_csv(CSV_FILE)
print("Rows and columns:", faqs.shape)
display(faqs.head())

## 3. Clean the data

In [ ]:
def clean_faqs(data):
    cleaned = data.copy()
    for column in ["question", "answer", "category"]:
        cleaned[column] = cleaned[column].astype("string").str.strip()
    cleaned = cleaned.dropna(subset=["question", "answer", "category"])
    cleaned = cleaned[
        (cleaned["question"] != "") &
        (cleaned["answer"] != "") &
        (cleaned["category"] != "")
    ]
    cleaned = cleaned.drop_duplicates(subset=["question"], keep="first")
    return cleaned.reset_index(drop=True)

faqs = clean_faqs(faqs)
print("FAQs after cleaning:", len(faqs))
display(faqs.head(3))

## 4. Create text chunks

Each short FAQ is one chunk: the question and answer stay together.

In [ ]:
def make_chunks(data):
    chunks = []
    for _, row in data.iterrows():
        chunks.append(f"Question: {row['question']}\\nAnswer: {row['answer']}")
    return chunks

chunks = make_chunks(faqs)
print("Number of chunks:", len(chunks))
print("\\nExample chunk:\\n")
print(chunks[0])

## 5. Create embeddings and store them

An embedding represents text as numbers. Chroma uses the embedding function for both stored FAQs and search queries. The first run may download the model.

In [ ]:
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="phone_repair_faqs",
    embedding_function=embedding_function,
)
ids = [f"faq-{i:03d}" for i in range(len(faqs))]
metadata = [
    {"question": row["question"], "category": row["category"], "answer": row["answer"]}
    for _, row in faqs.iterrows()
]
collection.upsert(ids=ids, documents=chunks, metadatas=metadata)
print("FAQs stored:", collection.count())

## 6. Search for the top three matches

Distance is a vector-distance measure, not a probability. A smaller distance usually indicates a closer match, but the result still needs to be checked.

In [ ]:
def search_faqs(query, collection, limit=3):
    if not query.strip():
        print("Please enter a question.")
        return
    count = collection.count()
    if count == 0:
        print("The FAQ collection is empty.")
        return
    results = collection.query(
        query_texts=[query],
        n_results=min(limit, count),
        include=["documents", "metadatas", "distances"],
    )
    print(f"Your question: {query}\\n")
    for rank, (document, metadata, distance) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        print(f"{rank}. {metadata['question']} ({metadata['category']})")
        print(metadata["answer"])
        print(f"Distance: {distance:.4f}\\n")

search_faqs("Why does my battery lose power so fast?", collection)

## 7. Try another query and record what you notice

In [ ]:
search_faqs("My phone is dead and will not start.", collection)

In [ ]:
# Replace this note after running the searches and checking the results.
observation = "I will describe what I noticed from my actual search results."
print(observation)

## 8. Conclusion

The project demonstrates CSV loading, cleaning, chunking, embeddings, vector storage, and semantic retrieval. Search quality depends on the FAQ data and embedding model. Run every cell from top to bottom and write your own observation before submitting.